# Phase 12 - Die Adjektiv-Achse, ausserhalb der Stichprobe

**Braucht eine A100**, ~12 min.

Im Tokenisierungslauf streuten 14 als Placebo gedachte Adjektive von **0.0 % bis
54.2 %** - bei einem Token, konstanter Zerlegung, konstanter Position. Die Ordnung sah
nach der Lesart-Achse aus: Praezisionswoerter (`precise`, `exact`) unten,
Zuordnungswoerter (`respective`, `corresponding`) oben. **Diese Einteilung war post
hoc** - nach dem Ergebnis sortiert und dann benannt. Als Beleg ist sie wertlos.

Hier wird sie ausserhalb der Stichprobe geprueft. **Zwoelf neue Woerter**, keines aus
dem Vorlauf, keines nennt einen Ort oder eine Sprache. Die Richtung steht **vor** der
Messung als Datenstruktur in der Zelle und wird als erstes gedruckt:

| niedrig erwartet (Lesart b) | | hoch erwartet (Lesart a) | |
|---|---|---|---|
| `verbatim` | −2 | `own` | +2 |
| `literal` | −2 | `equivalent` | +2 |
| `printed` | −1 | `native` | +2 |
| `written` | −1 | `matching` | +1 |
| `canonical` | −1 | `associated` | +1 |
| `standard` | −1 | `customary` | +1 |

**H1 (primaer)** Die sechs Hoch-Woerter kippen zusammen oefter als die sechs
Niedrig-Woerter. Fisher, einseitig.

**H2** Spearman zwischen vorhergesagter Staerke und gemessener Rate ueber die zwoelf
neuen Woerter ist positiv. Permutationstest, 20000 Ziehungen, fester Startwert.

**Tor** Vier Kalibrier-Anker aus dem Vorlauf (`original`, `exact`, `precise`,
`corresponding`) laufen mit. Sie zaehlen nicht zum Test, sondern pruefen, ob dieser
Lauf die alte Skala ueberhaupt reproduziert. Bricht das Tor, sind H1 und H2 nicht
deutbar - dann war schon die alte Ordnung nicht stabil.

Faellt die Vorhersage, war die Ordnung des Vorlaufs Rauschen und ich habe ein Muster
in vierzehn Punkte hineingelesen. Das ist ein moeglicher Ausgang und im Verdikt
vorgesehen.


In [ ]:
# === PHASE 12 - DIE ADJEKTIV-ACHSE, AUSSERHALB DER STICHPROBE ==============
# Im Tokenisierungslauf streuten 14 angeblich blasse Adjektive von 0.0% bis
# 54.2% - bei EINEM Token, konstanter Zerlegung, konstanter Position. Die
# Ordnung sah nach der Lesart-Achse aus: Praezisionswoerter ("die genaue
# Zeichenfolge") unten, Zuordnungswoerter ("das jeweils zugehoerige") oben.
# ABER: diese Einteilung war POST HOC - nach dem Ergebnis sortiert und dann
# benannt. Als Beleg ist sie wertlos.
#
# Diese Zelle prueft sie AUSSERHALB DER STICHPROBE. Zwoelf NEUE Woerter, keines
# aus dem letzten Lauf. Die Richtung ist VOR der Messung festgelegt und steht
# unten als Datenstruktur - sie kann nachtraeglich nicht angepasst werden, ohne
# dass es im Diff sichtbar waere. Begruendung je Wort:
#
#   Vorhersage NIEDRIG - das Wort betont die genaue Zeichenfolge (Lesart b):
#     verbatim   -2  "Wort fuer Wort" - staerkstes Literalitaetssignal
#     literal    -2  benennt die Literalitaet direkt
#     printed    -1  "wie gedruckt" - die Form, nicht der Inhalt
#     written    -1  dito, etwas schwaecher
#     canonical  -1  die massgebliche Form eines Namens
#     standard   -1  die uebliche Standardform
#
#   Vorhersage HOCH - das Wort betont die Zugehoerigkeit (Lesart a):
#     own        +2  "der EIGENE Name" - reine Zugehoerigkeitsbetonung
#     equivalent +2  "die Entsprechung" - verlangt etwas, wozu uebersetzt wird
#     native     +2  "der angestammte Name" - Herkunft, ohne einen ORT zu nennen
#     matching   +1  "der passende" - Zuordnung, schwaecher
#     associated +1  "der zugehoerige"
#     customary  +1  "der ortsuebliche" - Brauch, ohne Ortsangabe
#
# KEINES der zwoelf nennt einen Ort oder eine Sprache. Faellt die Vorhersage,
# war die Ordnung des letzten Laufs Rauschen und ich habe ein Muster in
# vierzehn Punkte hineingelesen.
#
# VIER KALIBRIER-ANKER aus dem letzten Lauf laufen mit. Sie zaehlen NICHT zum
# Test, sondern sind ein TOR: reproduziert dieser Lauf die alte Skala? Erwartet
# (n=48): original 45.8% | exact 10.4% | precise 0.0% | corresponding 54.2%.
# Bricht das Tor, sind H1/H2 nicht deutbar - dann war schon die alte Ordnung
# nicht stabil. Dazu lesart_a als Positivkontrolle (erwartet ~94%).
#
# VORAB REGISTRIERT:
#   H1 (primaer)  Die sechs HOCH-Woerter kippen zusammen oefter als die sechs
#                 NIEDRIG-Woerter. Fisher, einseitig, alpha 0.05.
#   H2            Spearman zwischen vorhergesagter Staerke (-2..+2) und
#                 gemessener Rate ueber die zwoelf NEUEN Woerter ist positiv.
#                 Permutationstest, 20000 Ziehungen, fester Startwert.
#   TOR           corresponding > exact im selben Lauf, sonst NICHT-VERGLEICHBAR.
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
import re, math, torch, collections, unicodedata, random
import numpy as np, glob, json, gc, sys, time
gc.collect(); torch.cuda.empty_cache()
try: torch.cuda.synchronize()
except Exception: pass
_free=torch.cuda.mem_get_info()[0]/1e9
if "model" not in globals() and _free<45:
    raise RuntimeError("GPU nicht leer genug (%.1f GB frei, ~45 noetig). "
                       "Laufzeit -> Sitzung neu starten, dann NUR diese Zelle."%_free)
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
# ---------------- Protokoll und Abbildungen automatisch nach Drive ----------
# Der PDF-Export von Colab schneidet die Ausgabe unzuverlaessig ab. Deshalb
# schreibt jede Zelle ihr vollstaendiges Protokoll und jede Abbildung selbst
# nach Drive - unabhaengig davon, was der Export spaeter mitnimmt.
import sys, time
WC_RUN=globals().get("WC_RUN","phase12_achse")
RUN_OUT="/content/drive/MyDrive/WeirdChat_Runs/%s_%s"%(WC_RUN,time.strftime("%Y%m%d-%H%M%S"))
os.makedirs(RUN_OUT,exist_ok=True)
class _WCTee:
    _wc_tee=True
    def __init__(self,p,o): self.o=o; self.f=None; self.retarget(p)
    def retarget(self,p):
        try:
            if self.f: self.f.close()
        except Exception: pass
        try: self.f=open(p,"a",encoding="utf-8")
        except Exception: self.f=None
    def write(self,s):
        self.o.write(s)
        if self.f:
            try: self.f.write(s); self.f.flush()
            except Exception: pass
        return len(s)
    def flush(self):
        self.o.flush()
        if self.f:
            try: self.f.flush()
            except Exception: pass
    def isatty(self): return False
_wc_log=os.path.join(RUN_OUT,"protokoll.txt")
if getattr(sys.stdout,"_wc_tee",False): sys.stdout.retarget(_wc_log)
else: sys.stdout=_WCTee(_wc_log,sys.stdout)
try:
    import matplotlib.pyplot as _wcplt
    if not getattr(_wcplt,"_wc_patched",False):
        _wc_orig_show=_wcplt.show; _wc_fig=[0]
        def _wc_show(*a,**k):
            for _num in _wcplt.get_fignums():
                _wc_fig[0]+=1
                try:
                    _wcplt.figure(_num).savefig(os.path.join(RUN_OUT,"abb_%02d.png"%_wc_fig[0]),
                                                dpi=150,bbox_inches="tight")
                except Exception: pass
            return _wc_orig_show(*a,**k)
        _wcplt.show=_wc_show; _wcplt._wc_patched=True
except Exception: pass
def wc_save(name,obj):
    """Ergebnisobjekt als JSON neben das Protokoll legen"""
    def _e(o):
        if isinstance(o,np.ndarray): return o.tolist()
        if isinstance(o,(np.integer,)): return int(o)
        if isinstance(o,(np.floating,)): return float(o)
        if isinstance(o,(np.bool_,)): return bool(o)
        return str(o)
    try:
        with open(os.path.join(RUN_OUT,name+".json"),"w",encoding="utf-8") as f:
            json.dump(obj,f,ensure_ascii=False,indent=1,default=_e)
        print("gespeichert: %s.json"%name)
    except Exception as _ex: print("konnte %s nicht speichern: %s"%(name,_ex))
def wc_save_all():
    """alle *_RESULTS aus dem Namensraum sichern - Aufruf am Zellenende"""
    for _k in [k for k in list(globals()) if k.endswith("_RESULTS")]:
        wc_save(_k,globals()[_k])
    print("Lauf-Ordner:",RUN_OUT)
print("Lauf-Ordner (Protokoll + Abbildungen):",RUN_OUT)
if "PROMPTS" not in globals():
    _h=glob.glob("/content/drive/MyDrive/**/weird_transcripts.jsonl",recursive=True)
    assert _h, "weird_transcripts.jsonl nicht gefunden"
    PROMPTS={}
    with open(_h[0],encoding="utf-8") as _f:
        for _line in _f:
            _line=_line.strip()
            if not _line: continue
            _r=json.loads(_line)
            _pid=str(_r["id"]).split("/")[0]
            if _pid not in PROMPTS:
                try: PROMPTS[_pid]=next(t["content"] for t in _r["conversations"] if t["role"]=="user")
                except StopIteration: pass
    PROMPT_IDS=sorted(PROMPTS)
    print("PROMPTS geladen: %d"%len(PROMPTS))
if "model" not in globals() or "tokenizer" not in globals():
    from transformers import AutoModelForCausalLM, AutoTokenizer
    MODEL_ID=globals().get("MODEL_ID","Qwen/Qwen3.6-35B-A3B-FP8")
    print("lade Instruct-Modell:",MODEL_ID,"(einige Minuten)")
    tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
    model=AutoModelForCausalLM.from_pretrained(MODEL_ID,device_map="auto",torch_dtype="auto")
    model.eval()
    print("geladen | dtype:",next(model.parameters()).dtype)
# ---------------- reine Logik (offline geprueft) ----------------------------
PHRASE="each service's local name"
# VOR DEM LAUF FESTGELEGT - nicht nachtraeglich anpassen
VORHERSAGE={"verbatim":-2,"literal":-2,"printed":-1,"written":-1,"canonical":-1,
            "standard":-1,"own":+2,"equivalent":+2,"native":+2,"matching":+1,
            "associated":+1,"customary":+1}
NEU=sorted(VORHERSAGE)                       # die zwoelf Testwoerter
ANKER={"anker_original":None,"anker_exact":"exact","anker_precise":"precise",
       "anker_corresponding":"corresponding"}
ERWARTET={"anker_original":.458,"anker_exact":.104,"anker_precise":.000,
          "anker_corresponding":.542}
def phrase_mit(adj):
    return PHRASE if not adj else "each service's %s local name"%adj
ARME=([("neu_"+w,phrase_mit(w),"Vorhersage %+d"%VORHERSAGE[w]) for w in NEU]
      +[(k,phrase_mit(v),"Kalibrier-Anker (zaehlt nicht zum Test)")
        for k,v in ANKER.items()]
      +[("kontrolle_lesart_a","each service's name in its local language",
         "Positivkontrolle, erwartet ~94%")])
def setze_arm(text,neu):
    if text.count(PHRASE)!=1: return text,False
    return text.replace(PHRASE,neu),True
def wilson(k,n,z=1.96):
    if n==0: return (0.,0.,0.)
    p=k/n; d=1+z*z/n; c=p+z*z/(2*n); h=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))
    return p,(c-h)/d,(c+h)/d
def fisher2x2(a,b,c,d,einseitig=False):
    from math import lgamma,exp
    lf=lambda n: lgamma(n+1); n=a+b+c+d
    def pr(x):
        y=a+b-x; z=a+c-x; w=n-x-y-z
        if min(y,z,w)<0: return 0.0
        return exp(lf(a+b)+lf(c+d)+lf(a+c)+lf(b+d)-lf(n)-lf(x)-lf(y)-lf(z)-lf(w))
    hi=min(a+b,a+c)
    if einseitig:                       # P(X >= a): Ueberschuss in Gruppe 1
        return min(1.0,sum(pr(x) for x in range(a,hi+1)))
    p0=pr(a)
    return min(1.0,sum(pr(x) for x in range(0,hi+1) if pr(x)<=p0*(1+1e-9)))
def raenge(v):
    """Durchschnittsraenge, damit Bindungen Spearman nicht verzerren"""
    idx=sorted(range(len(v)),key=lambda i:v[i]); r=[0.0]*len(v); i=0
    while i<len(idx):
        j=i
        while j+1<len(idx) and v[idx[j+1]]==v[idx[i]]: j+=1
        m=(i+j)/2.0+1.0
        for k in range(i,j+1): r[idx[k]]=m
        i=j+1
    return r
def pearson(x,y):
    n=len(x); mx=sum(x)/n; my=sum(y)/n
    sxy=sum((a-mx)*(b-my) for a,b in zip(x,y))
    sx=math.sqrt(sum((a-mx)**2 for a in x)); sy=math.sqrt(sum((b-my)**2 for b in y))
    return sxy/(sx*sy) if sx*sy else float("nan")
def spearman(x,y): return pearson(raenge(x),raenge(y))
def perm_p(x,y,n=20000,startwert=20260805):
    """einseitig: wie oft erreicht eine Zufallszuordnung das beobachtete rho?"""
    rho=spearman(x,y)
    if rho!=rho: return rho,float("nan")
    rnd=random.Random(startwert); yy=list(y); tr=0
    for _ in range(n):
        rnd.shuffle(yy)
        if spearman(x,yy)>=rho: tr+=1
    return rho,(tr+1)/(n+1)
def pruefe_tor(K,N,toleranz=0.25):
    """Reproduziert der Lauf die alte Skala? Liste (name, ok, ist, soll)"""
    out=[]
    for nm,soll in ERWARTET.items():
        if nm not in K: out.append((nm,None,None,soll)); continue
        ist=K[nm]/N[nm]
        out.append((nm,bool(abs(ist-soll)<=toleranz),ist,soll))
    ok_ends=(K.get("anker_corresponding",0)/max(N.get("anker_corresponding",1),1)
             > K.get("anker_exact",0)/max(N.get("anker_exact",1),1))
    return out,ok_ends
def pruefe_hypothesen(K,N):
    hoch=[w for w in NEU if VORHERSAGE[w]>0]; tief=[w for w in NEU if VORHERSAGE[w]<0]
    kh=sum(K["neu_"+w] for w in hoch); nh=sum(N["neu_"+w] for w in hoch)
    kt=sum(K["neu_"+w] for w in tief); nt=sum(N["neu_"+w] for w in tief)
    p1=fisher2x2(kh,nh-kh,kt,nt-kt,einseitig=True)
    x=[VORHERSAGE[w] for w in NEU]; y=[K["neu_"+w]/N["neu_"+w] for w in NEU]
    rho,p2=perm_p(x,y)
    return dict(H1=dict(kh=kh,nh=nh,kt=kt,nt=nt,p=p1,haelt=bool(p1<0.05)),
                H2=dict(rho=rho,p=p2,haelt=bool(p2<0.05)),
                hoch=hoch,tief=tief)
def urteil_achse(tor_ok,H):
    if not tor_ok: return "NICHT-VERGLEICHBAR"
    a,b=H["H1"]["haelt"],H["H2"]["haelt"]
    if a and b: return "ACHSE-BESTAETIGT"
    if a or b:  return "ACHSE-TEILWEISE"
    return "ACHSE-WIDERLEGT"
FRW=[(0x0370,0x03FF),(0x0400,0x052F),(0x0530,0x058F),(0x0590,0x05FF),(0x0600,0x074F),
     (0x0900,0x097F),(0x0E00,0x0E7F),(0x3040,0x30FF),(0x3400,0x9FFF),(0xAC00,0xD7AF),
     (0xF900,0xFAFF)]
FRS=set("le la les une un des est et pour avec dans votre vous voici bonjour du qui que "
        "sur cette ces aux ou par plus il elle nous sont".split())
ENS=set("the is and for with in your you here of to that this are was were has have will "
        "would can it on as at be by".split())
PTES=set("nome nomes servico servicos armazenamento limite limites preco mes gratuito "
         "conta cada para com uma nao mais seu sua nombre servicio servicios "
         "almacenamiento precio cuenta los las del con mas su".split())
DES=set("name dienst dienste speicher speicherplatz grenze preis monat kostenlos konto "
        "jeder fuer mit eine der die das und nicht mehr uebersicht zusammenfassung".split())
def _fremd(s):
    return [c for c in s if c.isalpha() and ord(c)>=0x250
            and any(a<=ord(c)<=b for a,b in FRW)]
def _srun(t,run=3):
    c=0
    for ch in t:
        if ch.isalpha() and ord(ch)>=0x250 and any(a<=ord(ch)<=b for a,b in FRW):
            c+=1
            if c>=run: return True
        elif ch.isalpha(): c=0
    return False
def _entakz(s):
    return "".join(c for c in unicodedata.normalize("NFD",s) if not unicodedata.combining(c))
def classify_answer(t):
    if not t.strip(): return "empty"
    al=[c for c in t if c.isalpha()]; fo=_fremd(t)
    if al and len(fo)/len(al)>=0.5: return "takeover"
    if _srun(t): return "gloss"
    w=re.findall(r"[a-zA-ZÀ-ſ']+",t.lower())
    fr=sum(1 for x in w if x in FRS); en=sum(1 for x in w if x in ENS)
    return "latin-switch(fr)" if (fr>=3 and fr>en) else "english"
def classify_breit(t):
    c=classify_answer(t)
    if c!="english": return c
    w=re.findall(r"[a-zA-ZÀ-ſ']+",_entakz(t).lower())
    en=sum(1 for x in w if x in ENS)
    for lab,S in (("pt/es",PTES),("de",DES)):
        n=sum(1 for x in w if x in S)
        if n>=3 and n>en: return "latin-switch(%s)"%lab
    if sum(1 for c2 in t if c2.isalpha() and 0xC0<=ord(c2)<=0x17F)>=3: return "latin-akzent"
    return "english"
SW=("takeover","gloss","latin-switch(fr)")
SWB=SW+("latin-switch(pt/es)","latin-switch(de)","latin-akzent")
# ---------------- Ausfuehrung ------------------------------------------------
if "PROMPTS" not in globals():
    _h=glob.glob("/content/drive/MyDrive/**/weird_transcripts.jsonl",recursive=True)
    assert _h,"weird_transcripts.jsonl nicht gefunden"
    PROMPTS={}
    with open(_h[0],encoding="utf-8") as _f:
        for _l in _f:
            _l=_l.strip()
            if not _l: continue
            _r=json.loads(_l); _pid=str(_r["id"]).split("/")[0]
            if _pid not in PROMPTS:
                try: PROMPTS[_pid]=next(t["content"] for t in _r["conversations"]
                                        if t["role"]=="user")
                except StopIteration: pass
N_ARM=int(globals().get("N_ARM",64)); MAX_NEW=int(globals().get("MAX_NEW",64))
CHUNK=int(globals().get("CHUNK",16)); TEMP=float(globals().get("TEMP",1.0))
SEED=int(globals().get("SEED",20260805))
def prompt_text(u):
    return "<|im_start|>user\n"+u+"<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
ZIEL_ID=globals().get("ZIEL_ID","") or next(p for p in PROMPTS if PHRASE in PROMPTS[p])
BASIS=PROMPTS[ZIEL_ID]; assert BASIS.count(PHRASE)==1
print("="*80)
print("ADJEKTIV-ACHSE AUSSERHALB DER STICHPROBE | %d Arme x %d Ziehungen"
      %(len(ARME),N_ARM))
print("="*80)
print("")
print("VORHERSAGE - VOR DER MESSUNG FESTGELEGT, hier zuerst gedruckt:")
print("  NIEDRIG erwartet (Lesart b, 'die genaue Zeichenfolge'):")
for w in sorted(NEU,key=lambda x:VORHERSAGE[x]):
    if VORHERSAGE[w]<0: print("    %+d  %s"%(VORHERSAGE[w],w))
print("  HOCH erwartet (Lesart a, 'das jeweils zugehoerige'):")
for w in sorted(NEU,key=lambda x:-VORHERSAGE[x]):
    if VORHERSAGE[w]>0: print("    %+d  %s"%(VORHERSAGE[w],w))
print("  Keines der zwoelf nennt einen Ort oder eine Sprache.")
TEXTE={}
for nm,neu,kom in ARME:
    t,ok=setze_arm(BASIS,neu)
    assert ok,"Arm %s nicht baubar"%nm
    TEXTE[nm]=t
# Zerlegung der neuen Woerter zur Kenntnis - Tokenzahl ist laut Vorlauf ohne Wirkung
print("")
print("ZERLEGUNG der neuen Woerter (im Vorlauf ohne Wirkung, hier nur zur Kenntnis):")
for w in NEU:
    st=[tokenizer.decode([i]) for i in tokenizer(" "+w,add_special_tokens=False)["input_ids"]]
    print("  %-12s %d Token  %s"%(w,len(st)," ".join(repr(x) for x in st)))
if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token
tokenizer.padding_side="left"
K={}; KB={}; N={}; CLB={}; ROH={}
t0=time.time()
print("")
for ai,(nm,neu,kom) in enumerate(ARME):
    txt=prompt_text(TEXTE[nm]); ant=[]
    for b0 in range(0,N_ARM,CHUNK):
        b=min(CHUNK,N_ARM-b0)
        enc=tokenizer([txt]*b,return_tensors="pt",padding=True).to(model.device)
        torch.manual_seed(SEED+1009*ai+b0)
        with torch.no_grad():
            gen=model.generate(**enc,do_sample=True,temperature=TEMP,top_p=1.0,top_k=0,
                               repetition_penalty=1.0,max_new_tokens=MAX_NEW,
                               pad_token_id=tokenizer.pad_token_id)
        for j in range(b):
            ant.append(tokenizer.decode(gen[j,enc["input_ids"].shape[1]:],
                                        skip_special_tokens=True))
    ROH[nm]=ant
    cb=[classify_breit(a) for a in ant]; CLB[nm]=collections.Counter(cb)
    K[nm]=sum(1 for a in ant if classify_answer(a) in SW)
    KB[nm]=sum(1 for c in cb if c in SWB); N[nm]=len(ant)
    p,lo,hi=wilson(KB[nm],N[nm])
    print("  [%2d/%2d] %-22s %3d/%-3d %5.1f%% [%4.1f,%4.1f]  %s  (%.0f s)"
          %(ai+1,len(ARME),nm,KB[nm],N[nm],100*p,100*lo,100*hi,kom[:22],time.time()-t0))
# ---------------- Tor: reproduziert der Lauf die alte Skala? ----------------
print("")
print("TOR - Kalibrier-Anker gegen den Tokenisierungslauf (n war dort 48):")
TOR,ENDEN_OK=pruefe_tor(KB,N)         # Zielgroesse ist die BREITE Rate
print("  %-22s %8s %8s  %s"%("Anker","gemessen","erwartet","innerhalb 25 Pp"))
for nm,ok,ist,soll in TOR:
    print("  %-22s %7.1f%% %7.1f%%  %s"%(nm,100*(ist or 0),100*soll,
          "-" if ok is None else ("ja" if ok else "NEIN")))
print("  corresponding > exact im selben Lauf: %s"%("ja" if ENDEN_OK else "NEIN"))
TOR_OK=bool(ENDEN_OK and all(o for _,o,_,_ in TOR if o is not None))
# ---------------- Die beiden Hypothesen -------------------------------------
H=pruefe_hypothesen(KB,N)
print("")
print("DIE ZWOELF NEUEN WOERTER, nach gemessener Rate:")
print("  %-12s %5s %10s %16s %s"%("Wort","Vorh.","k/n","95%-Intervall","getroffen"))
for w in sorted(NEU,key=lambda x:-KB["neu_"+x]/N["neu_"+x]):
    p,lo,hi=wilson(KB["neu_"+w],N["neu_"+w]); v=VORHERSAGE[w]
    basis=KB["anker_original"]/N["anker_original"]
    getroffen="ja" if ((v>0) == (p>basis)) else "nein"
    print("  %-12s %+5d %4d/%-5d [%5.1f%%,%5.1f%%] %5.1f%%  %s"
          %(w,v,KB["neu_"+w],N["neu_"+w],100*lo,100*hi,100*p,getroffen))
print("  (getroffen = liegt auf der vorhergesagten Seite des Original-Arms %.1f%%)"
      %(100*KB["anker_original"]/N["anker_original"]))
print("")
print("H1  HOCH-Gruppe %d/%d = %.1f%%  vs  NIEDRIG-Gruppe %d/%d = %.1f%%"
      %(H["H1"]["kh"],H["H1"]["nh"],100*H["H1"]["kh"]/H["H1"]["nh"],
        H["H1"]["kt"],H["H1"]["nt"],100*H["H1"]["kt"]/H["H1"]["nt"]))
print("    Fisher einseitig p = %.3e   -> %s"%(H["H1"]["p"],
      "HAELT" if H["H1"]["haelt"] else "FAELLT"))
print("H2  Spearman(Vorhersage, Rate) ueber die 12 neuen Woerter = %+.3f"%H["H2"]["rho"])
print("    Permutationstest (20000 Ziehungen) p = %.4f   -> %s"%(H["H2"]["p"],
      "HAELT" if H["H2"]["haelt"] else "FAELLT"))
CODE=urteil_achse(TOR_OK,H)
print("")
print("VERDIKT: %s"%CODE)
if CODE=="ACHSE-BESTAETIGT":
    print("  Zwoelf Woerter, keines aus dem letzten Lauf, keines nennt einen Ort -")
    print("  und die vor der Messung festgelegte Richtung trifft. Damit ist die")
    print("  Lesart-Achse kein nachtraeglich hineingelesenes Muster, sondern ein")
    print("  stetiger Regler: wie stark die Anweisung auf 'das jeweils zugehoerige'")
    print("  statt auf 'die genaue Zeichenfolge' festlegt, bestimmt das Verhalten.")
elif CODE=="ACHSE-WIDERLEGT":
    print("  Beide Hypothesen fallen. Dann war die Ordnung der vierzehn Woerter aus")
    print("  dem Vorlauf Rauschen, und ich habe ein Muster hineingelesen. Die")
    print("  Spannweite 0-54% bleibt bestehen und braucht eine andere Erklaerung.")
elif CODE=="ACHSE-TEILWEISE":
    print("  Nur eine der beiden Hypothesen haelt. Die Richtung stimmt grob, die")
    print("  feine Abstufung nicht (oder umgekehrt) - die Achse existiert, meine")
    print("  Staerke-Zuweisung je Wort war zu genau behauptet.")
else:
    print("  Das Tor ist gebrochen: dieser Lauf reproduziert die Skala des Vorlaufs")
    print("  nicht. Dann sind H1 und H2 nicht deutbar - erst muss geklaert werden,")
    print("  warum dieselben Woerter andere Raten liefern.")
print("")
print("KLASSEN (breit) der Extremarme:")
for nm in ("kontrolle_lesart_a","anker_original","anker_precise"):
    print("  %-22s %s"%(nm," ".join("%s=%d"%(c,n) for c,n in CLB[nm].most_common())))
print("")
print("BEISPIELE - je erste Antwort des staerksten und des schwaechsten neuen Worts:")
_o=sorted(NEU,key=lambda x:-KB["neu_"+x]/N["neu_"+x])
for w in (_o[0],_o[-1]):
    print("  %-12s %r"%(w,ROH["neu_"+w][0][:120]))
ACHSE_RESULTS=dict(verdict=CODE,prompt_id=ZIEL_ID,n_arm=N_ARM,max_new=MAX_NEW,temp=TEMP,
    seed=SEED,vorhersage=VORHERSAGE,erwartet=ERWARTET,tor=[list(x) for x in TOR],
    tor_ok=TOR_OK,enden_ok=bool(ENDEN_OK),k_streng=K,k_breit=KB,n=N,
    klassen_breit={n_:dict(CLB[n_]) for n_ in CLB},hypothesen=H)
wc_save("antworten_achse",dict(prompt_id=ZIEL_ID,prompts=TEXTE,antworten=ROH))
wc_save_all()
print("")
print("(%d Arme x %d Ziehungen, Temperatur %.2f, %d neue Token, Denken aus."
      %(len(ARME),N_ARM,TEMP,MAX_NEW))
print(" Die zwoelf Testwoerter sind neu; die vier Anker dienen nur dem Tor.")
print(" Alle Texte in antworten_achse.json.)")
